# Cuantización, LoRA y pruebas de carga

Notebook opcional del **módulo 22** del curso (en la app: `uv run dev` y la ruta `#/modulo/despliegue`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

> ⚠️ **Requiere GPU** con CUDA para ejecutarse en un tiempo razonable.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

nombre = "Qwen/Qwen2.5-1.5B"
tok = AutoTokenizer.from_pretrained(nombre)
fp16 = AutoModelForCausalLM.from_pretrained(nombre, torch_dtype=torch.float16, device_map="cuda")
nf4 = AutoModelForCausalLM.from_pretrained(nombre, device_map="cuda",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
for nombre_m, m in [("FP16", fp16), ("NF4", nf4)]:
    print(nombre_m, round(m.get_memory_footprint() / 1e9, 2), "GB")

In [ ]:
texto = "The KV cache stores keys and values of previous tokens so that each new token only needs to compute its own projections." * 4
x = tok(texto, return_tensors="pt").to("cuda")
for nombre_m, m in [("FP16", fp16), ("NF4", nf4)]:
    with torch.no_grad():
        print(nombre_m, "perplejidad", round(torch.exp(m(**x, labels=x["input_ids"]).loss).item(), 3))

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, task_type="CAUSAL_LM")
lora = get_peft_model(nf4, config)          # QLoRA: base en 4 bits + adaptadores en 16 bits
lora.print_trainable_parameters()

**Prueba de carga** contra un servidor compatible con la API de OpenAI (por ejemplo `vllm serve Qwen/Qwen2.5-1.5B-Instruct`): mide TTFT, latencia entre tokens y percentiles con concurrencia creciente.

In [ ]:
import asyncio
import time

import numpy as np
from openai import AsyncOpenAI

cliente = AsyncOpenAI(base_url="http://localhost:8000/v1", api_key="local")


async def una_peticion():
    t0 = time.perf_counter()
    ttft = None
    stream = await cliente.chat.completions.create(model="Qwen/Qwen2.5-1.5B-Instruct", stream=True, max_tokens=128,
                                                   messages=[{"role": "user", "content": "Explica la KV cache en tres frases."}])
    async for trozo in stream:
        if ttft is None:
            ttft = time.perf_counter() - t0
    return ttft, time.perf_counter() - t0


async def carga(concurrencia):
    r = np.array(await asyncio.gather(*[una_peticion() for _ in range(concurrencia)]))
    print(concurrencia, "TTFT p50/p95", np.percentile(r[:, 0], [50, 95]).round(2), "total p95", np.percentile(r[:, 1], 95).round(2))

# for c in [1, 8, 32]: await carga(c)